In [1]:
!pip -q install pandas numpy tqdm rdflib gensim
# RDKit 建议用 conda 安装；如果你 notebook 环境已装 RDKit 可跳过
# !conda install -y -c conda-forge rdkit


In [2]:
import pandas as pd
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, OWL
from rdkit import Chem

ELEMENTKG_OWL = "./KANO/KGembedding/elementkg.owl"   # 你上传/保存的 elementkg.owl 路径
OUT_TSV = "./FG_SMARTS_ElementKG.tsv"

g = Graph()
g.parse(ELEMENTKG_OWL)

base = "http://www.semanticweb.org/ElementKG#"
hasBondType = URIRef(base + "hasBondType")

# 83 个 FG 个体：满足 “有 hasBondType” + 特例 Sulfonate（你这个 owl 里它没 hasBondType）
named = set(g.subjects(RDF.type, OWL.NamedIndividual))
fg_inds = [ind for ind in named if str(ind).startswith(base) and (ind, hasBondType, None) in g]
fg_inds.append(URIRef(base + "Sulfonate"))  # 补上
fg_names = sorted(set(str(ind).split("#")[-1] for ind in fg_inds))
print("[INFO] FG individuals =", len(fg_names))
print(fg_names)

# === 全量 83 个 FG 的 SMARTS 映射（与 fg_names 严格对齐；无缺失） ===
FG_SMARTS = {
"4ammoniumIon":"[NX4+]",
"Acetal":"[CX4H1]([OX2][#6])([OX2][#6])[#6]",
"Aldehyde":"[CX3H1](=O)[#6]",
"Azo":"[NX2]=[NX2]",
"Azide":"[N-]=[N+]=N",
"Amidine":"[CX3](=[NX3])[NX3]",
"Alkenyl":"C=C",
"Alkynyl":"C#C",
"Alkyl":"[CX4]",
"Alkylaluminium":"[Al][CX4]",
"Alkyllithium":"[Li][CX4]",
"AlkylmagnesiumHalide":"[Mg]([F,Cl,Br,I])[CX4]",
"Borono":"[BX3]([OX2H])([OX2H])",
"Boronate":"[BX3]([OX2][#6])([OX2][#6])",
"Borinate":"[BX3]([OX2][#6])([OX2][#6])",
"Borino":"[BX3]",
"Carbamate":"[NX3][CX3](=O)[OX2][#6]",
"Carboalkoxy":"[CX3](=O)[OX2][#6]",
"Carbodithio":"[CX3](=S)[SX2][#6]",
"CarbodithioicAcid":"[CX3](=S)[SX2H]",
"CarbonateEster":"[OX2][CX3](=O)[OX2]",
"Carbonyl":"[CX3]=[OX1]",
"CarbothioicOAcid":"[CX3](=S)[OX2H]",
"CarbothioicSAcid":"[CX3](=O)[SX2H]",
"Carboxamide":"[CX3](=O)[NX3]",
"Carboxyl":"[CX3](=O)[OX2H]",
"Carboxylate":"[CX3](=O)[O-]",
"CarboxylicAnhydride":"[CX3](=O)O[CX3](=O)",
"Cyanate":"[OX2][CX2]#N",
"Disulfide":"[SX2][SX2]",
"Ether":"[OD2]([#6])[#6]",
"Haloformyl":"[CX3](=O)[F,Cl,Br,I]",
"Hemiacetal":"[CX4H1]([OX2H])([OX2][#6])[#6]",
"Hemiketal":"[CX4H0]([OX2H])([OX2][#6])([#6])[#6]",
"Hydroperoxy":"[OX2][OX2H]",
"Hydroxyl":"[OX2H]",
"Imide":"[NX3]([CX3](=O))[CX3](=O)",
"Isocyanate":"N=C=O",
"Isonitrile":"[NX1]#[CX2]",
"Isothiocyanate":"N=C=S",
"Ketal":"[CX4H0]([OX2][#6])([OX2][#6])([#6])[#6]",
"Methylenedioxy":"[OX2][CH2][OX2]",
"Nitrate":"[OX2][NX3](=O)=O",
"Nitrile":"[CX2]#N",
"Nitro":"[NX3](=O)=O",
"Nitroso":"[NX2]=O",
"Nitrosooxy":"[NX2](=O)[OX2]",
"OrthocarbonateEster":"[CX4]([OX2][#6])([OX2][#6])([OX2][#6])[OX2][#6]",
"Orthoester":"[CX4]([OX2][#6])([OX2][#6])[OX2][#6]",
"Oxime":"[CX3](=N[OX2H])",
"Peroxy":"[OX2][OX2]",
"Phenyl":"c1ccccc1",
"Phosphate":"[PX4](=O)([OX2H,OX1-])([OX2H,OX1-])[OX2H,OX1-]",
"Phosphino":"[PX3]([#6])([#6])[#6]",
"Phosphodiester":"[PX4](=O)([OX2][#6])([OX2][#6])[OX2H,OX1-]",
"Phosphono":"[PX4](=O)([OX2H,OX1-])([OX2H,OX1-])[#6]",
"PrimaryAldimine":"[CX3H1](=N[H])[#6]",
"SecondaryAldimine":"[CX3H1](=N[#6])[#6]",
"PrimaryKetimine":"[CX3]([#6])([#6])=N[H]",
"SecondaryKetimine":"[CX3]([#6])([#6])=N[#6]",
"PrimaryAmine":"[NX3H2][#6]",
"SecondaryAmine":"[NX3H1]([#6])[#6]",
"TertiaryAmine":"[NX3H0]([#6])([#6])[#6]",
"Pyridyl":"n1ccccc1",
"SilylEther":"[OX2][Si]",
"Sulfhydryl":"[SX2H]",
"Sulfide":"[SX2]([#6])[#6]",
"Sulfino":"[SX3](=O)[OX2H]",
"Sulfinyl":"[SX3](=O)[#6]",
"Sulfo":"[SX4](=O)(=O)[OX2H]",
"Sulfoate":"[SX4](=O)(=O)[OX2][#6]",
"Sulfonate":"[SX4](=O)(=O)[O-]",
"Sulfonyl":"[SX4](=O)(=O)[#6]",
"Thial":"[CX3H1](=S)[#6]",
"Thiocyanate":"[SX2][CX2]#N",
"Thioketone":"[CX3](=S)[#6]",
"Thiolester":"[CX3](=O)[SX2][#6]",
"Thionoester":"[CX3](=S)[OX2][#6]",
"bromo":"[Br]",
"chloro":"[Cl]",
"fluoro":"[F]",
"halo":"[F,Cl,Br,I]",
"iodo":"[I]",
}

# 校验：必须完全覆盖 83 个 FG 个体
missing = set(fg_names) - set(FG_SMARTS.keys())
extra = set(FG_SMARTS.keys()) - set(fg_names)
assert not missing, f"FG_SMARTS 缺失这些 key：{sorted(missing)}"
assert not extra, f"FG_SMARTS 多了这些 key：{sorted(extra)}"

# 校验：SMARTS 必须能被 RDKit 解析
bad = []
for k,v in FG_SMARTS.items():
    if Chem.MolFromSmarts(v) is None:
        bad.append((k,v))
assert not bad, f"这些 SMARTS RDKit 解析失败：{bad}"

# 输出 TSV
rows = []
for name in fg_names:
    rows.append({
        "fg_name": name,
        "iri": base + name,
        "smarts": FG_SMARTS[name],
    })
df = pd.DataFrame(rows)
df.to_csv(OUT_TSV, sep="\t", index=False)
print("[OK] wrote:", OUT_TSV)
df.head(10)


[INFO] FG individuals = 83
['4ammoniumIon', 'Acetal', 'Aldehyde', 'Alkenyl', 'Alkyl', 'Alkylaluminium', 'Alkyllithium', 'AlkylmagnesiumHalide', 'Alkynyl', 'Amidine', 'Azide', 'Azo', 'Borinate', 'Borino', 'Boronate', 'Borono', 'Carbamate', 'Carboalkoxy', 'Carbodithio', 'CarbodithioicAcid', 'CarbonateEster', 'Carbonyl', 'CarbothioicOAcid', 'CarbothioicSAcid', 'Carboxamide', 'Carboxyl', 'Carboxylate', 'CarboxylicAnhydride', 'Cyanate', 'Disulfide', 'Ether', 'Haloformyl', 'Hemiacetal', 'Hemiketal', 'Hydroperoxy', 'Hydroxyl', 'Imide', 'Isocyanate', 'Isonitrile', 'Isothiocyanate', 'Ketal', 'Methylenedioxy', 'Nitrate', 'Nitrile', 'Nitro', 'Nitroso', 'Nitrosooxy', 'OrthocarbonateEster', 'Orthoester', 'Oxime', 'Peroxy', 'Phenyl', 'Phosphate', 'Phosphino', 'Phosphodiester', 'Phosphono', 'PrimaryAldimine', 'PrimaryAmine', 'PrimaryKetimine', 'Pyridyl', 'SecondaryAldimine', 'SecondaryAmine', 'SecondaryKetimine', 'SilylEther', 'Sulfhydryl', 'Sulfide', 'Sulfino', 'Sulfinyl', 'Sulfo', 'Sulfoate', 'Sulf

,fg_name,iri,smarts
0,4ammoniumIon,http://www.semanticweb.org/ElementKG#4ammoniumIon,[NX4+]
1,Acetal,http://www.semanticweb.org/ElementKG#Acetal,[CX4H1]([OX2][#6])([OX2][#6])[#6]
2,Aldehyde,http://www.semanticweb.org/ElementKG#Aldehyde,[CX3H1](=O)[#6]
3,Alkenyl,http://www.semanticweb.org/ElementKG#Alkenyl,C=C
4,Alkyl,http://www.semanticweb.org/ElementKG#Alkyl,[CX4]
5,Alkylaluminium,http://www.semanticweb.org/ElementKG#Alkylalum...,[Al][CX4]
6,Alkyllithium,http://www.semanticweb.org/ElementKG#Alkyllithium,[Li][CX4]
7,AlkylmagnesiumHalide,http://www.semanticweb.org/ElementKG#Alkylmagn...,"[Mg]([F,Cl,Br,I])[CX4]"
8,Alkynyl,http://www.semanticweb.org/ElementKG#Alkynyl,C#C
9,Amidine,http://www.semanticweb.org/ElementKG#Amidine,[CX3](=[NX3])[NX3]


In [1]:
import os, json, re, random
import numpy as np
import pandas as pd
from tqdm import tqdm

STRUCTKG_OWL = r"./StructKG.owl"               # 你的本体
DATA_PATH    = r"./Malodors_data.xlsx"         # 你的 SMILES 数据
SMILES_COL   = "nonStereoSMILES"              # 你的 SMILES 列名（改成你的）
ID_COL       = None                            # 可选：id 列名，不用就 None

# 可选：FunctionalGroup SMARTS 映射（强烈建议你用“与 KANO 一致”的那份）
# TSV 格式：fg_name <TAB> smarts
FG_SMARTS_TSV = r"./FG_SMARTS_ElementKG.tsv"         # 没有就设 None（但这样 FG 就无法从 SMILES 投影！）

OUT_DIR = "./StructKG_ontoemb_out"
os.makedirs(OUT_DIR, exist_ok=True)

# ====== 训练 KG embeddings（对齐 KANO：owl 图->walk语料->word2vec）======
EMBED_DIM      = 200
WALK_LENGTH    = 16
WALKS_PER_NODE = 20
WINDOW         = 5
NEGATIVE       = 10
EPOCHS         = 10
SEED           = 42

# walk 语料里是否插入关系 token（rel:xxx）
USE_PREDICATE_TOKENS = True

# SMILES->prompt token 最大长度（KANO 的 prompt 也会截断）
MAX_PROMPT_LEN = 64


In [7]:
# from rdflib import Graph, URIRef
# from rdflib.namespace import RDF, RDFS, OWL

# def local_name(iri: str) -> str:
#     return iri.split("#")[-1] if "#" in iri else iri.rstrip("/").split("/")[-1]

# g = Graph()
# g.parse(STRUCTKG_OWL)
# print("[INFO] triples:", len(g))

# # ---- subclass 图（用于找 Element/FunctionalGroup/Scaffold 子类闭包）----
# sub_adj = {}
# for s,o in g.subject_objects(RDFS.subClassOf):
#     if isinstance(s, URIRef) and isinstance(o, URIRef):
#         sub_adj.setdefault(str(o), set()).add(str(s))

# def subclasses_of(root_iri: str):
#     # BFS closure
#     seen = set()
#     q = [root_iri]
#     while q:
#         x = q.pop()
#         for ch in sub_adj.get(x, []):
#             if ch not in seen:
#                 seen.add(ch)
#                 q.append(ch)
#     return seen

# # 找到顶层类 IRI：Element / FunctionalGroup / Scaffold（按 local name 精确匹配）
# def find_class_iri_by_name(name: str):
#     for c in g.subjects(RDF.type, OWL.Class):
#         if isinstance(c, URIRef) and local_name(str(c)) == name:
#             return str(c)
#     # 有的本体用 rdfs:Class
#     for c in g.subjects(RDF.type, RDFS.Class):
#         if isinstance(c, URIRef) and local_name(str(c)) == name:
#             return str(c)
#     return None

# ELEMENT_CLASS = find_class_iri_by_name("Element")
# FG_CLASS      = find_class_iri_by_name("FunctionalGroup")
# SCAF_CLASS    = find_class_iri_by_name("Scaffold")

# print("[INFO] Element class:", ELEMENT_CLASS)
# print("[INFO] FunctionalGroup class:", FG_CLASS)
# print("[INFO] Scaffold class:", SCAF_CLASS)

# assert ELEMENT_CLASS and FG_CLASS and SCAF_CLASS, "找不到 Element/FunctionalGroup/Scaffold 顶层类，请检查 OWL 里的类名"

# element_subs = subclasses_of(ELEMENT_CLASS) | {ELEMENT_CLASS}
# fg_subs      = subclasses_of(FG_CLASS)      | {FG_CLASS}
# scaf_subs    = subclasses_of(SCAF_CLASS)    | {SCAF_CLASS}

# # ---- 个体：owl:NamedIndividual 或者有 rdf:type 的资源 ----
# named_inds = set(str(s) for s in g.subjects(RDF.type, OWL.NamedIndividual))
# typed_inds = set(str(s) for s,_ in g.subject_objects(RDF.type) if isinstance(s, URIRef))
# inds = named_inds | typed_inds

# # rdf:type 记录
# types_of = {}
# for s,o in g.subject_objects(RDF.type):
#     if isinstance(s, URIRef) and isinstance(o, URIRef):
#         types_of.setdefault(str(s), set()).add(str(o))

# # Element/FG/Scaffold 个体划分：看其 rdf:type 是否属于对应子类集合
# element_entities = {}
# fg_entities = {}
# scaf_entities = {}

# for ind in inds:
#     tset = types_of.get(ind, set())
#     if tset & element_subs:
#         element_entities[local_name(ind)] = ind
#     if tset & fg_subs:
#         fg_entities[local_name(ind)] = ind
#     if tset & scaf_subs:
#         scaf_entities[local_name(ind)] = ind

# print("[INFO] Element individuals:", len(element_entities))
# print("[INFO] FG individuals:", len(fg_entities))
# print("[INFO] Scaffold individuals:", len(scaf_entities))
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, RDFS, OWL

STRUCTKG_OWL = r"./StructKG.owl"   # 改成你的路径

def local_name(iri: str) -> str:
    return iri.split("#")[-1] if "#" in iri else iri.rstrip("/").split("/")[-1]

g = Graph()
g.parse(STRUCTKG_OWL)
print("[INFO] triples:", len(g))

# ---- 1) 找类 IRI：大小写不敏感匹配 ----
def find_class_iri_ci(target_name: str):
    target = target_name.lower()
    cand = set()
    for c in g.subjects(RDF.type, OWL.Class):
        if isinstance(c, URIRef) and local_name(str(c)).lower() == target:
            cand.add(str(c))
    for c in g.subjects(RDF.type, RDFS.Class):
        if isinstance(c, URIRef) and local_name(str(c)).lower() == target:
            cand.add(str(c))
    return sorted(cand)

ELEMENT_CANDS = find_class_iri_ci("element")
FG_CANDS      = find_class_iri_ci("functionalGroup")
SCAF_CANDS    = find_class_iri_ci("scaffold")

print("[INFO] Element class candidates:", ELEMENT_CANDS[:5])
print("[INFO] FunctionalGroup class candidates:", FG_CANDS[:5])
print("[INFO] Scaffold class candidates:", SCAF_CANDS[:5])

assert len(ELEMENT_CANDS) >= 1, "仍然找不到 element（注意：你的 OWL 里可能叫 elements/Element 等，发我类名我给你适配）"
assert len(FG_CANDS) >= 1, "仍然找不到 functionalGroup"
assert len(SCAF_CANDS) >= 1, "仍然找不到 Scaffold"

# 默认取第一个（一般只有一个）
ELEMENT_CLASS = ELEMENT_CANDS[0]
FG_CLASS      = FG_CANDS[0]
SCAF_CLASS    = SCAF_CANDS[0]

print("[OK] use Element:", ELEMENT_CLASS)
print("[OK] use FG:", FG_CLASS)
print("[OK] use Scaffold:", SCAF_CLASS)

# ---- 2) subclass 闭包 ----
sub_adj = {}
for s,o in g.subject_objects(RDFS.subClassOf):
    if isinstance(s, URIRef) and isinstance(o, URIRef):
        sub_adj.setdefault(str(o), set()).add(str(s))

def subclasses_of(root_iri: str):
    seen = set()
    stack = [root_iri]
    while stack:
        x = stack.pop()
        for ch in sub_adj.get(x, []):
            if ch not in seen:
                seen.add(ch)
                stack.append(ch)
    return seen

element_subs = subclasses_of(ELEMENT_CLASS) | {ELEMENT_CLASS}
fg_subs      = subclasses_of(FG_CLASS)      | {FG_CLASS}
scaf_subs    = subclasses_of(SCAF_CLASS)    | {SCAF_CLASS}

print("[INFO] #Element subclasses:", len(element_subs))
print("[INFO] #FG subclasses:", len(fg_subs))
print("[INFO] #Scaffold subclasses:", len(scaf_subs))

# ---- 3) individuals 抽取（按 rdf:type 是否落在对应子类闭包）----
named_inds = set(str(s) for s in g.subjects(RDF.type, OWL.NamedIndividual))
typed_inds = set(str(s) for s,_ in g.subject_objects(RDF.type) if isinstance(s, URIRef))
inds = named_inds | typed_inds

types_of = {}
for s,o in g.subject_objects(RDF.type):
    if isinstance(s, URIRef) and isinstance(o, URIRef):
        types_of.setdefault(str(s), set()).add(str(o))

element_entities, fg_entities, scaf_entities = {}, {}, {}
for ind in inds:
    tset = types_of.get(ind, set())
    if tset & element_subs:
        element_entities[local_name(ind)] = ind
    if tset & fg_subs:
        fg_entities[local_name(ind)] = ind
    if tset & scaf_subs:
        scaf_entities[local_name(ind)] = ind

print("[INFO] Element individuals:", len(element_entities))
print("[INFO] FG individuals:", len(fg_entities))
print("[INFO] Scaffold individuals:", len(scaf_entities))


[INFO] triples: 62725
[INFO] Element class candidates: ['http://www.semanticweb.org/StructKG#element']
[INFO] FunctionalGroup class candidates: ['http://www.semanticweb.org/StructKG#functionalGroup']
[INFO] Scaffold class candidates: ['http://www.semanticweb.org/StructKG#Scaffold']
[OK] use Element: http://www.semanticweb.org/StructKG#element
[OK] use FG: http://www.semanticweb.org/StructKG#functionalGroup
[OK] use Scaffold: http://www.semanticweb.org/StructKG#Scaffold
[INFO] #Element subclasses: 27
[INFO] #FG subclasses: 80
[INFO] #Scaffold subclasses: 71
[INFO] Element individuals: 118
[INFO] FG individuals: 82
[INFO] Scaffold individuals: 100


In [8]:
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

def canon_smiles_no_kek(smi: str):
    try:
        m = Chem.MolFromSmiles(smi, sanitize=False)
        if m is None:
            return None
        ops = Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE
        Chem.SanitizeMol(m, sanitizeOps=ops)
        return Chem.MolToSmiles(m, canonical=True, isomericSmiles=False)
    except Exception:
        return None

# 找到 hasRingSystemSMILES 这个 data property
hasRingSystemSMILES = None
for p in set(g.predicates()):
    if isinstance(p, URIRef) and local_name(str(p)) == "hasRingSystemSMILES":
        hasRingSystemSMILES = str(p)
        break
assert hasRingSystemSMILES, "OWl里找不到 hasRingSystemSMILES data property"

scaf_smiles2iri = {}
for s,o in g.subject_objects(URIRef(hasRingSystemSMILES)):
    if isinstance(s, URIRef):
        smi = str(o).strip()
        cs = canon_smiles_no_kek(smi) or smi
        scaf_smiles2iri[cs] = str(s)

print("[INFO] scaffold SMILES->IRI:", len(scaf_smiles2iri))


[INFO] scaffold SMILES->IRI: 100


In [9]:
from gensim.models import Word2Vec

random.seed(SEED)
np.random.seed(SEED)

# 我们把 Class + Individual 都作为节点；把 subClassOf / rdf:type / 所有 ObjectProperty 当作边
# 这跟 KANO/ElementKG 的 owl2vec 思路一致：本体图越完整，embedding 越“本体化”。

# 识别 ObjectProperty
obj_props = set(str(p) for p in g.subjects(RDF.type, OWL.ObjectProperty))

def add_edge(adj, a, rel, b):
    adj.setdefault(a, []).append((rel, b))
    adj.setdefault(b, []).append((rel, a))  # 当作无向用于 walk（ElementKG 常见处理）

adj = {}
nodes = set()

# subClassOf edges
for s,o in g.subject_objects(RDFS.subClassOf):
    if isinstance(s, URIRef) and isinstance(o, URIRef):
        a, b = str(s), str(o)
        add_edge(adj, a, "rel:subClassOf", b)
        nodes.add(a); nodes.add(b)

# rdf:type edges (instance->class)
for s,o in g.subject_objects(RDF.type):
    if isinstance(s, URIRef) and isinstance(o, URIRef):
        a, b = str(s), str(o)
        add_edge(adj, a, "rel:type", b)
        nodes.add(a); nodes.add(b)

# all object property edges
for s,p,o in g.triples((None, None, None)):
    if not (isinstance(s, URIRef) and isinstance(p, URIRef) and isinstance(o, URIRef)):
        continue
    if str(p) not in obj_props:
        continue
    a, b = str(s), str(o)
    rel = "rel:" + local_name(str(p))
    add_edge(adj, a, rel, b)
    nodes.add(a); nodes.add(b)

print("[INFO] nodes for embedding:", len(nodes))
print("[INFO] approx undirected edges:", sum(len(v) for v in adj.values())//2)

def random_walk(start, length):
    walk = [start]
    cur = start
    for _ in range(length-1):
        nbrs = adj.get(cur, [])
        if not nbrs:
            break
        rel, nxt = random.choice(nbrs)
        if USE_PREDICATE_TOKENS:
            walk.append(rel)
        walk.append(nxt)
        cur = nxt
    return walk

# build corpus
nodes_list = list(nodes)
corpus = []
for n in tqdm(nodes_list, desc="Build walk corpus"):
    for _ in range(WALKS_PER_NODE):
        corpus.append(random_walk(n, WALK_LENGTH))

print("[INFO] corpus size:", len(corpus))

# train w2v
w2v = Word2Vec(
    sentences=corpus,
    vector_size=EMBED_DIM,
    window=WINDOW,
    min_count=1,
    sg=1,
    negative=NEGATIVE,
    workers=max(1, os.cpu_count() or 1),
    seed=SEED
)
w2v.train(corpus, total_examples=len(corpus), epochs=EPOCHS)

emb_txt = os.path.join(OUT_DIR, "structkgontology.embeddings.txt")
w2v.wv.save_word2vec_format(emb_txt, binary=False)
print("[OK] saved:", emb_txt)


[INFO] nodes for embedding: 595
[INFO] approx undirected edges: 58631


Build walk corpus: 100%|██████████| 595/595 [00:00<00:00, 4495.47it/s]

[INFO] corpus size: 11900


[OK] saved: ./StructKG_ontoemb_out/structkgontology.embeddings.txt


In [10]:
from gensim.models import KeyedVectors

kv = KeyedVectors.load_word2vec_format(emb_txt, binary=False)
dim = kv.vector_size
print("[INFO] loaded embeddings vocab:", len(kv.key_to_index), "dim:", dim)

# 只保留“实体 token”（Element + FG + Scaffold 个体），关系 token rel:* 不进入 prompt
entity_iris = list(element_entities.values()) + list(fg_entities.values()) + list(scaf_entities.values())

# embeddings 里 token 就是 IRI 字符串（我们训练时用的就是 IRI）
kept = [t for t in entity_iris if t in kv]

token2id = {t:i for i,t in enumerate(kept)}
emb_mat = np.vstack([kv[t] for t in kept]).astype(np.float32)

with open(os.path.join(OUT_DIR, "structkgontology.token2id.json"), "w", encoding="utf-8") as f:
    json.dump(token2id, f, ensure_ascii=False, indent=2)
np.save(os.path.join(OUT_DIR, "structkgontology.embeddings.npy"), emb_mat)

print("[OK] kept entity tokens:", len(kept))
print("[OK] wrote token2id + embeddings.npy")


[INFO] loaded embeddings vocab: 694 dim: 200
[OK] kept entity tokens: 300
[OK] wrote token2id + embeddings.npy


In [11]:
def load_fg_smarts_tsv(path):
    df = pd.read_csv(path, sep="\t")
    # 兼容两种：带表头 fg_name/smarts 或无表头
    if df.shape[1] >= 2 and ("fg_name" in df.columns and "smarts" in df.columns):
        names = df["fg_name"].astype(str).tolist()
        smarts= df["smarts"].astype(str).tolist()
    else:
        df = pd.read_csv(path, sep="\t", header=None, names=["fg_name","smarts"])
        names = df["fg_name"].astype(str).tolist()
        smarts= df["smarts"].astype(str).tolist()

    mp = {}
    for n,s in zip(names, smarts):
        n = n.strip(); s = s.strip()
        if not n or not s:
            continue
        # 只保留 OWL 里真的存在的 FG 个体（完全对齐 ElementKG 风格）
        if n in fg_entities:
            patt = Chem.MolFromSmarts(s)
            if patt is not None:
                mp[n] = patt
    return mp

assert FG_SMARTS_TSV is not None and os.path.exists(FG_SMARTS_TSV), \
    "你必须提供 FG_SMARTS_TSV（与 KANO/ElementKG 一致的FG SMARTS映射），否则无法投影 FG tokens"

fg_smarts = load_fg_smarts_tsv(FG_SMARTS_TSV)
print("[INFO] FG SMARTS compiled:", len(fg_smarts), "| OWL FG entities:", len(fg_entities))


[INFO] FG SMARTS compiled: 82 | OWL FG entities: 82


In [14]:
def ring_systems_atoms(mol: Chem.Mol):
    ri = mol.GetRingInfo()
    rings = [set(r) for r in ri.AtomRings()]
    if not rings:
        return []
    changed = True
    while changed:
        changed = False
        new = []
        used = [False]*len(rings)
        for i in range(len(rings)):
            if used[i]:
                continue
            cur = set(rings[i])
            used[i] = True
            for j in range(i+1, len(rings)):
                if used[j]:
                    continue
                if cur & rings[j]:
                    cur |= rings[j]
                    used[j] = True
                    changed = True
            new.append(cur)
        rings = new
    return [sorted(list(s)) for s in rings]

def submol_smiles_no_kek(mol, atom_idx):
    try:
        smi = Chem.MolFragmentToSmiles(mol, atomsToUse=atom_idx, canonical=True, isomericSmiles=False)
        return canon_smiles_no_kek(smi) or smi
    except Exception:
        return None

def smiles_to_tokens_iri(smi: str):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return []

    toks = []

    # 1) Element tokens：用元素符号映射到 OWL element 个体 IRI
    elems = sorted({a.GetSymbol() for a in m.GetAtoms()})
    for e in elems:
        if e in element_entities:
            toks.append(element_entities[e])

    # 2) FG tokens：SMARTS 命中 -> OWL FG 个体 IRI
    for fg_name, patt in fg_smarts.items():
        try:
            if m.HasSubstructMatch(patt):
                toks.append(fg_entities[fg_name])
        except Exception:
            pass

    # 3) Scaffold tokens：ring-system smiles -> OWL Scaffold 个体 IRI
    rs_atoms_list = ring_systems_atoms(m)
    rs_smiles = []
    for atoms in rs_atoms_list:
        s = submol_smiles_no_kek(m, atoms)
        if s:
            rs_smiles.append(s)
    rs_smiles = sorted(set(rs_smiles))
    for rs in rs_smiles:
        if rs in scaf_smiles2iri:
            toks.append(scaf_smiles2iri[rs])

    # 去重保序 + 截断
    seen=set(); out=[]
    for t in toks:
        if t not in seen:
            seen.add(t); out.append(t)
    return out[:MAX_PROMPT_LEN]

# 读取数据
df = pd.read_excel(DATA_PATH) if DATA_PATH.lower().endswith((".xlsx",".xls")) else pd.read_csv(DATA_PATH)
assert SMILES_COL in df.columns, f"找不到 SMILES 列：{SMILES_COL}"
smiles_list = df[SMILES_COL].astype(str).fillna("").tolist()

# embeddings 查表（我们用 token2id + emb_mat）
token2id = json.load(open(os.path.join(OUT_DIR, "structkgontology.token2id.json"), "r", encoding="utf-8"))
emb_mat = np.load(os.path.join(OUT_DIR, "structkgontology.embeddings.npy"))
dim = emb_mat.shape[1]

rows_tokens = []
rows_ids = []
mol_emb = np.zeros((len(smiles_list), dim), dtype=np.float32)

miss_stats = {"miss_element":0, "miss_fg":0, "miss_scaf":0, "empty":0}

for i,smi in enumerate(tqdm(smiles_list, desc="Project SMILES")):
    cs = canon_smiles_no_kek(smi)
    if cs is None:
        rows_tokens.append("")
        rows_ids.append("")
        miss_stats["empty"] += 1
        continue

    toks = smiles_to_tokens_iri(cs)

    # 统计缺失情况（命中 token2id 的才算有 embedding）
    ids = []
    for t in toks:
        if t in token2id:
            ids.append(token2id[t])

    rows_tokens.append(" ".join([local_name(t) for t in toks]))
    rows_ids.append(" ".join(map(str, ids)))

    if ids:
        mol_emb[i] = emb_mat[ids].mean(axis=0)
    else:
        miss_stats["empty"] += 1

print("[INFO] miss stats:", miss_stats)

out = df.copy()
out["KG_tokens"] = rows_tokens
out["KG_token_ids"] = rows_ids
for j in range(dim):
    out[f"KG_emb_{j:03d}"] = mol_emb[:, j]

out_xlsx = os.path.join(OUT_DIR, "SMILES_StructKG_prompt_embeddings.xlsx")
out.to_excel(out_xlsx, index=False)
np.save(os.path.join(OUT_DIR, "SMILES_StructKG_prompt_embeddings.npy"), mol_emb)
print("[OK] saved:", out_xlsx)


Project SMILES: 100%|██████████| 4952/4952 [00:02<00:00, 2138.97it/s]
/tmp/ipykernel_71434/1751769483.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"KG_emb_{j:03d}"] = mol_emb[:, j]
/tmp/ipykernel_71434/1751769483.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"KG_emb_{j:03d}"] = mol_emb[:, j]
/tmp/ipykernel_71434/1751769483.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column

[INFO] miss stats: {'miss_element': 0, 'miss_fg': 0, 'miss_scaf': 0, 'empty': 0}
[OK] saved: ./StructKG_ontoemb_out/SMILES_StructKG_prompt_embeddings.xlsx
